# SIC 3674 SEC Dataset Expansion — Google Colab

This notebook expands the research dataset from a small hand-picked semiconductor sample to the **SEC SIC 3674 (Semiconductors & Related Devices)** universe.

It:

- discovers SIC 3674 registrants from the SEC bulk submissions archive;
- separates domestic 10-Q/10-K issuers from foreign-private-issuer filing patterns;
- downloads SEC Company Facts;
- reconstructs quarterly revenue and operating cash flow where needed;
- creates gross-margin, inventory, receivables, and cash-flow features;
- creates an 8-quarter lag history;
- creates the next-quarter **revenue-growth momentum** target;
- preserves accession/filed metadata for the later text-only vs. fundamentals-only vs. combined experiment;
- writes CSV, Parquet, and DuckDB outputs under `/content/sic3674_output`.

Run the cells from top to bottom.


## 1. Install packages and set the SEC User-Agent

SEC asks automated clients to identify themselves. Replace the placeholder email in the next cell before running the data download.


In [ ]:
!pip -q install duckdb pyarrow pandas numpy requests beautifulsoup4 lxml


In [ ]:
# REQUIRED: identify yourself to SEC.gov with a descriptive User-Agent.
# Replace the example below with your name/project and a real contact email.
import os

os.environ["SEC_USER_AGENT"] = "Your Name SIC3674 research your-email@example.com"

print("SEC_USER_AGENT =", os.environ["SEC_USER_AGENT"])


## 2. Dataset construction functions

This cell contains the reusable SIC-universe, SEC Company Facts, quarterly reconstruction, feature-engineering, and lag-generation functions.


In [ ]:
from __future__ import annotations

import io
import json
import os
import re
import time
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ---------------------------
# Configuration
# ---------------------------

SIC_CODE = 3674
OUTPUT_DIR = Path("/content/sic3674_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEC_USER_AGENT = os.environ["SEC_USER_AGENT"]

SUBMISSIONS_ZIP_URL = (
    "https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip"
)
COMPANYFACTS_URL = (
    "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"
)

# SEC fair-access guidance is <=10 requests/sec. Stay comfortably below it.
REQUEST_SLEEP_SECONDS = 0.15

# Primary paper sample: domestic issuers with standardized 10-Q/10-K reporting.
# Foreign private issuers remain in the universe/audit files and can be handled
# as a later robustness extension.
DOMESTIC_FORMS = {"10-Q", "10-Q/A", "10-K", "10-K/A"}
FOREIGN_FORMS = {"20-F", "20-F/A", "40-F", "40-F/A", "6-K"}

# Eight prior quarters are required by the model design.
LOOKBACK_QUARTERS = 8

# Neutral-band sensitivity values for future growth-momentum classification.
NEUTRAL_BANDS = (0.00, 0.01, 0.02, 0.05)

# XBRL tag candidates. We choose the tag with the best usable coverage per firm.
REVENUE_TAGS = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
    "SalesRevenueNet",
    "SalesRevenueGoodsNet",
]

GROSS_PROFIT_TAGS = [
    "GrossProfit",
]

INVENTORY_TAGS = [
    "InventoryNet",
    "InventoryNetOfAllowancesCustomerAdvancesAndProgressBillings",
]

AR_TAGS = [
    "AccountsReceivableNetCurrent",
    "AccountsReceivableNet",
    "AccountsNotesAndLoansReceivableNetCurrent",
]

OCF_TAGS = [
    "NetCashProvidedByUsedInOperatingActivities",
    "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations",
]


# ---------------------------
# Networking
# ---------------------------

def make_session() -> requests.Session:
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": SEC_USER_AGENT,
            "Accept-Encoding": "gzip, deflate",
        }
    )
    retry = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


def get_bytes(session: requests.Session, url: str) -> bytes:
    # Host header must match data.sec.gov when that endpoint is used.
    headers = {"User-Agent": SEC_USER_AGENT}
    response = session.get(url, headers=headers, timeout=120)
    response.raise_for_status()
    return response.content


def get_json(session: requests.Session, url: str) -> dict:
    time.sleep(REQUEST_SLEEP_SECONDS)
    headers = {"User-Agent": SEC_USER_AGENT}
    response = session.get(url, headers=headers, timeout=90)
    response.raise_for_status()
    return response.json()


# ---------------------------
# Universe construction
# ---------------------------

def normalize_cik(value) -> str:
    return str(int(value)).zfill(10)


def flatten_recent_filings(submission: dict) -> pd.DataFrame:
    recent = submission.get("filings", {}).get("recent", {})
    if not recent:
        return pd.DataFrame()

    lengths = [
        len(v) for v in recent.values()
        if isinstance(v, list)
    ]
    if not lengths:
        return pd.DataFrame()

    n = min(lengths)
    payload = {
        key: value[:n]
        for key, value in recent.items()
        if isinstance(value, list)
    }
    return pd.DataFrame(payload)


def issuer_form_family(forms: Iterable[str]) -> str:
    forms = set(str(x) for x in forms)
    if forms & {"10-Q", "10-Q/A"}:
        return "domestic_10q_10k"
    if forms & {"20-F", "20-F/A", "40-F", "40-F/A"}:
        return "foreign_private_issuer"
    return "other_or_unknown"


def build_sic_universe_from_submissions(zip_bytes: bytes) -> pd.DataFrame:
    rows = []

    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        # Only top-level CIK##########.json files. Supplemental history JSONs
        # have different names and should not be treated as separate entities.
        cik_pattern = re.compile(r"^CIK\d{10}\.json$")

        for name in zf.namelist():
            base = Path(name).name
            if not cik_pattern.match(base):
                continue

            try:
                submission = json.loads(zf.read(name))
            except Exception:
                continue

            sic_raw = submission.get("sic")
            try:
                sic = int(sic_raw)
            except (TypeError, ValueError):
                continue

            if sic != SIC_CODE:
                continue

            recent = flatten_recent_filings(submission)
            forms = recent["form"].astype(str).tolist() if "form" in recent else []

            tickers = submission.get("tickers") or []
            exchanges = submission.get("exchanges") or []

            rows.append(
                {
                    "cik": normalize_cik(submission.get("cik")),
                    "company": submission.get("name"),
                    "sic": sic,
                    "sic_description": submission.get("sicDescription"),
                    "entity_type": submission.get("entityType"),
                    "fiscal_year_end": submission.get("fiscalYearEnd"),
                    "state_of_incorporation": submission.get("stateOfIncorporation"),
                    "tickers": "|".join(map(str, tickers)),
                    "exchanges": "|".join(map(str, exchanges)),
                    "currently_has_ticker": bool(tickers),
                    "issuer_form_family": issuer_form_family(forms),
                    "recent_10q_count": sum(f in {"10-Q", "10-Q/A"} for f in forms),
                    "recent_10k_count": sum(f in {"10-K", "10-K/A"} for f in forms),
                    "recent_20f_count": sum(f in {"20-F", "20-F/A"} for f in forms),
                    "recent_6k_count": sum(f == "6-K" for f in forms),
                }
            )

    universe = pd.DataFrame(rows)
    if universe.empty:
        raise RuntimeError(
            "No SIC 3674 registrants were found. Check the SEC download and SIC field."
        )

    universe = (
        universe
        .drop_duplicates(subset=["cik"])
        .sort_values(["currently_has_ticker", "company"], ascending=[False, True])
        .reset_index(drop=True)
    )
    return universe


# ---------------------------
# Company Facts helpers
# ---------------------------

def _taxonomy_order(companyfacts: dict) -> list[str]:
    facts = companyfacts.get("facts", {})
    ordered = []
    for taxonomy in ("us-gaap", "ifrs-full"):
        if taxonomy in facts:
            ordered.append(taxonomy)
    ordered.extend(t for t in facts if t not in ordered)
    return ordered


def fact_rows_for_tag(
    companyfacts: dict,
    tag: str,
    unit_preferences: tuple[str, ...] = ("USD",),
) -> pd.DataFrame:
    facts = companyfacts.get("facts", {})

    for taxonomy in _taxonomy_order(companyfacts):
        tag_obj = facts.get(taxonomy, {}).get(tag)
        if not tag_obj:
            continue

        units = tag_obj.get("units", {})
        unit_key = next((u for u in unit_preferences if u in units), None)
        if unit_key is None:
            # Fallback: use first USD-like unit if present.
            unit_key = next(
                (u for u in units if str(u).upper() == "USD"),
                None,
            )
        if unit_key is None:
            continue

        frame = pd.DataFrame(units[unit_key])
        if frame.empty or "val" not in frame:
            continue

        frame["taxonomy"] = taxonomy
        frame["tag"] = tag
        frame["unit"] = unit_key
        return frame

    return pd.DataFrame()


def choose_best_tag_rows(
    companyfacts: dict,
    candidates: list[str],
    *,
    instant: bool = False,
) -> tuple[str | None, pd.DataFrame]:
    best_tag = None
    best = pd.DataFrame()
    best_score = -1

    for tag in candidates:
        frame = fact_rows_for_tag(companyfacts, tag)
        if frame.empty:
            continue

        frame = frame.copy()
        if "form" in frame:
            frame = frame[frame["form"].isin(DOMESTIC_FORMS)]

        if instant:
            score = frame["end"].nunique() if "end" in frame else 0
        else:
            if {"start", "end"}.issubset(frame.columns):
                start = pd.to_datetime(frame["start"], errors="coerce")
                end = pd.to_datetime(frame["end"], errors="coerce")
                duration = (end - start).dt.days
                score = frame.loc[duration.between(60, 420), "end"].nunique()
            else:
                score = 0

        if score > best_score:
            best_score = score
            best_tag = tag
            best = frame

    return best_tag, best


def _prepare_duration_rows(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty or not {"start", "end", "filed", "val", "form"}.issubset(frame.columns):
        return pd.DataFrame()

    x = frame.copy()
    x["start"] = pd.to_datetime(x["start"], errors="coerce")
    x["end"] = pd.to_datetime(x["end"], errors="coerce")
    x["filed"] = pd.to_datetime(x["filed"], errors="coerce")
    x["duration_days"] = (x["end"] - x["start"]).dt.days
    x["val"] = pd.to_numeric(x["val"], errors="coerce")
    x = x.dropna(subset=["start", "end", "filed", "val"])
    x = x[x["form"].isin(DOMESTIC_FORMS)]
    return x


def _first_public_observation(frame: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    """Choose the earliest filing for a repeated fact period."""
    if frame.empty:
        return frame
    sort_cols = group_cols + ["filed"]
    return (
        frame.sort_values(sort_cols)
        .drop_duplicates(subset=group_cols, keep="first")
        .reset_index(drop=True)
    )


def extract_quarterly_duration_fact(
    companyfacts: dict,
    tag_candidates: list[str],
    value_name: str,
) -> tuple[pd.DataFrame, str | None]:
    """
    Build standalone quarterly facts.

    Q1-Q3 normally come from 10-Q contexts of roughly one quarter.
    Q4 is derived as annual 10-K minus the three earlier standalone quarters.
    """
    tag, raw = choose_best_tag_rows(companyfacts, tag_candidates, instant=False)
    x = _prepare_duration_rows(raw)
    if x.empty:
        return pd.DataFrame(), tag

    direct = x[
        x["duration_days"].between(65, 115)
    ].copy()
    direct = _first_public_observation(direct, ["start", "end"])

    annual = x[
        x["duration_days"].between(300, 400)
    ].copy()
    annual = _first_public_observation(annual, ["start", "end"])

    out_rows = []

    for _, row in direct.iterrows():
        out_rows.append(
            {
                "quarter_start": row["start"],
                "quarter_end": row["end"],
                value_name: row["val"],
                f"{value_name}_filed": row["filed"],
                f"{value_name}_accn": row.get("accn"),
                f"{value_name}_form": row.get("form"),
                f"{value_name}_source": "direct_quarter",
                f"{value_name}_tag": tag,
            }
        )

    existing_ends = {pd.Timestamp(r["quarter_end"]) for r in out_rows}

    for _, year in annual.iterrows():
        year_start = year["start"]
        year_end = year["end"]

        if pd.Timestamp(year_end) in existing_ends:
            continue

        q = direct[
            (direct["start"] >= year_start - pd.Timedelta(days=7))
            & (direct["end"] < year_end)
            & (direct["end"] > year_start)
        ].copy()

        # Keep one row per quarter end within this fiscal year.
        q = q.sort_values("end").drop_duplicates("end", keep="first")

        # The latest three standalone quarters before fiscal year end
        # should correspond to Q1-Q3.
        if len(q) < 3:
            continue
        q = q.tail(3)

        q_sum = q["val"].sum()
        q4_value = year["val"] - q_sum

        q3_end = q["end"].max()
        q4_start = q3_end + pd.Timedelta(days=1)

        out_rows.append(
            {
                "quarter_start": q4_start,
                "quarter_end": year_end,
                value_name: q4_value,
                f"{value_name}_filed": year["filed"],
                f"{value_name}_accn": year.get("accn"),
                f"{value_name}_form": year.get("form"),
                f"{value_name}_source": "derived_q4_from_annual",
                f"{value_name}_tag": tag,
            }
        )

    if not out_rows:
        return pd.DataFrame(), tag

    out = pd.DataFrame(out_rows)
    out = (
        out.sort_values(["quarter_end", f"{value_name}_filed"])
        .drop_duplicates("quarter_end", keep="first")
        .reset_index(drop=True)
    )
    return out, tag


def extract_quarterly_cumulative_cashflow(
    companyfacts: dict,
    tag_candidates: list[str],
    value_name: str = "operating_cash_flow",
) -> tuple[pd.DataFrame, str | None]:
    """
    Cash-flow statements in 10-Q are usually year-to-date:
       Q1 = Q1 YTD
       Q2 = Q2 YTD - Q1 YTD
       Q3 = Q3 YTD - Q2 YTD
       Q4 = FY - Q3 YTD
    """
    tag, raw = choose_best_tag_rows(companyfacts, tag_candidates, instant=False)
    x = _prepare_duration_rows(raw)
    if x.empty:
        return pd.DataFrame(), tag

    annual = x[x["duration_days"].between(300, 400)].copy()
    annual = _first_public_observation(annual, ["start", "end"])

    ytd = x[x["duration_days"].between(60, 310)].copy()
    ytd = _first_public_observation(ytd, ["start", "end"])

    out_rows = []

    for _, year in annual.iterrows():
        ys, ye = year["start"], year["end"]

        candidates = ytd[
            (abs((ytd["start"] - ys).dt.days) <= 10)
            & (ytd["end"] < ye)
            & (ytd["end"] > ys)
        ].copy()

        if candidates.empty:
            continue

        # Map by cumulative duration.
        q1 = candidates[candidates["duration_days"].between(65, 115)]
        q2 = candidates[candidates["duration_days"].between(145, 220)]
        q3 = candidates[candidates["duration_days"].between(230, 310)]

        if q1.empty or q2.empty or q3.empty:
            continue

        q1 = q1.sort_values(["end", "filed"]).iloc[0]
        q2 = q2.sort_values(["end", "filed"]).iloc[0]
        q3 = q3.sort_values(["end", "filed"]).iloc[0]

        values = [
            (q1["end"], q1["val"], q1),
            (q2["end"], q2["val"] - q1["val"], q2),
            (q3["end"], q3["val"] - q2["val"], q3),
            (ye, year["val"] - q3["val"], year),
        ]

        starts = [
            ys,
            q1["end"] + pd.Timedelta(days=1),
            q2["end"] + pd.Timedelta(days=1),
            q3["end"] + pd.Timedelta(days=1),
        ]

        for start, (end, val, src) in zip(starts, values):
            out_rows.append(
                {
                    "quarter_start": start,
                    "quarter_end": end,
                    value_name: val,
                    f"{value_name}_filed": src["filed"],
                    f"{value_name}_accn": src.get("accn"),
                    f"{value_name}_form": src.get("form"),
                    f"{value_name}_source": (
                        "derived_from_ytd_or_annual"
                    ),
                    f"{value_name}_tag": tag,
                }
            )

    if not out_rows:
        return pd.DataFrame(), tag

    out = pd.DataFrame(out_rows)
    out = (
        out.sort_values(["quarter_end", f"{value_name}_filed"])
        .drop_duplicates("quarter_end", keep="first")
        .reset_index(drop=True)
    )
    return out, tag


def extract_instant_fact(
    companyfacts: dict,
    tag_candidates: list[str],
    value_name: str,
) -> tuple[pd.DataFrame, str | None]:
    tag, raw = choose_best_tag_rows(companyfacts, tag_candidates, instant=True)
    if raw.empty or not {"end", "filed", "val", "form"}.issubset(raw.columns):
        return pd.DataFrame(), tag

    x = raw.copy()
    x = x[x["form"].isin(DOMESTIC_FORMS)]
    x["end"] = pd.to_datetime(x["end"], errors="coerce")
    x["filed"] = pd.to_datetime(x["filed"], errors="coerce")
    x["val"] = pd.to_numeric(x["val"], errors="coerce")
    x = x.dropna(subset=["end", "filed", "val"])
    x = _first_public_observation(x, ["end"])

    out = pd.DataFrame(
        {
            "quarter_end": x["end"],
            value_name: x["val"],
            f"{value_name}_filed": x["filed"],
            f"{value_name}_accn": x.get("accn"),
            f"{value_name}_form": x.get("form"),
            f"{value_name}_source": "instant_balance_sheet",
            f"{value_name}_tag": tag,
        }
    )
    return out, tag


def merge_feature(base: pd.DataFrame, feature: pd.DataFrame) -> pd.DataFrame:
    if feature.empty:
        return base
    feature = feature.drop(columns=["quarter_start"], errors="ignore")
    return base.merge(feature, on="quarter_end", how="left")


def build_company_quarters(companyfacts: dict, universe_row: pd.Series) -> pd.DataFrame:
    revenue, revenue_tag = extract_quarterly_duration_fact(
        companyfacts, REVENUE_TAGS, "revenue"
    )
    if revenue.empty:
        return pd.DataFrame()

    gross_profit, _ = extract_quarterly_duration_fact(
        companyfacts, GROSS_PROFIT_TAGS, "gross_profit"
    )
    ocf, _ = extract_quarterly_cumulative_cashflow(
        companyfacts, OCF_TAGS, "operating_cash_flow"
    )
    inventory, _ = extract_instant_fact(
        companyfacts, INVENTORY_TAGS, "inventory"
    )
    ar, _ = extract_instant_fact(
        companyfacts, AR_TAGS, "accounts_receivable"
    )

    q = revenue.copy()
    q = merge_feature(q, gross_profit)
    q = merge_feature(q, ocf)
    q = merge_feature(q, inventory)
    q = merge_feature(q, ar)

    q["cik"] = universe_row["cik"]
    q["company"] = universe_row["company"]
    q["tickers"] = universe_row["tickers"]
    q["exchanges"] = universe_row["exchanges"]
    q["issuer_form_family"] = universe_row["issuer_form_family"]

    q = q.sort_values("quarter_end").reset_index(drop=True)

    # Drop impossible/duplicate values.
    q = q[np.isfinite(q["revenue"]) & (q["revenue"] > 0)].copy()
    q = q.drop_duplicates(["cik", "quarter_end"], keep="first")

    # Point-in-time availability: latest filing date among the features used.
    filed_cols = [
        c for c in q.columns
        if c.endswith("_filed")
    ]
    if filed_cols:
        q["availability_date"] = q[filed_cols].max(axis=1)
    else:
        q["availability_date"] = q["revenue_filed"]

    # Ratios / fundamentals.
    q["gross_margin"] = q["gross_profit"] / q["revenue"]
    q["ocf_margin"] = q["operating_cash_flow"] / q["revenue"]

    q["revenue_ttm"] = (
        q["revenue"]
        .rolling(4, min_periods=4)
        .sum()
    )
    q["inventory_to_quarterly_revenue"] = q["inventory"] / q["revenue"]
    q["receivables_to_quarterly_revenue"] = (
        q["accounts_receivable"] / q["revenue"]
    )
    q["inventory_ratio"] = q["inventory"] / q["revenue_ttm"]
    q["receivables_ratio"] = q["accounts_receivable"] / q["revenue_ttm"]

    # Growth and growth momentum.
    q["revenue_growth_yoy"] = q["revenue"] / q["revenue"].shift(4) - 1.0
    q["revenue_growth_change"] = (
        q["revenue_growth_yoy"] - q["revenue_growth_yoy"].shift(1)
    )

    # Future target is g_(t+1) - g_t.
    q["future_growth_change"] = (
        q["revenue_growth_yoy"].shift(-1) - q["revenue_growth_yoy"]
    )

    for band in NEUTRAL_BANDS:
        suffix = f"{int(round(band * 10000)):04d}bp"
        if band == 0:
            q[f"target_{suffix}"] = np.where(
                q["future_growth_change"].isna(),
                np.nan,
                np.where(q["future_growth_change"] > 0, 1, 0),
            )
        else:
            q[f"target_{suffix}"] = np.select(
                [
                    q["future_growth_change"] > band,
                    q["future_growth_change"] < -band,
                ],
                [1, -1],
                default=0,
            )
            q.loc[q["future_growth_change"].isna(), f"target_{suffix}"] = np.nan

    # Use revenue filing accession as the canonical filing-text join key.
    q["source_accession"] = q["revenue_accn"]
    q["source_form"] = q["revenue_form"]
    q["source_filing_date"] = q["revenue_filed"]

    return q


# ---------------------------
# Lag features / model-ready rows
# ---------------------------

CORE_NUMERIC_FEATURES = [
    "revenue_growth_yoy",
    "revenue_growth_change",
    "gross_margin",
    "inventory_ratio",
    "receivables_ratio",
    "ocf_margin",
]


def add_lag_features(all_quarters: pd.DataFrame) -> pd.DataFrame:
    all_quarters = (
        all_quarters
        .sort_values(["cik", "quarter_end"])
        .reset_index(drop=True)
    )

    pieces = []
    for cik, group in all_quarters.groupby("cik", sort=False):
        g = group.copy()
        for feature in CORE_NUMERIC_FEATURES:
            if feature not in g:
                continue
            for lag in range(LOOKBACK_QUARTERS):
                g[f"{feature}_lag{lag}"] = g[feature].shift(lag)

        revenue_lags = [
            f"revenue_growth_yoy_lag{i}"
            for i in range(LOOKBACK_QUARTERS)
        ]
        full_lags = [
            f"{feature}_lag{i}"
            for feature in CORE_NUMERIC_FEATURES
            for i in range(LOOKBACK_QUARTERS)
            if f"{feature}_lag{i}" in g.columns
        ]

        g["eligible_revenue_history"] = (
            g[revenue_lags].notna().all(axis=1)
            & g["future_growth_change"].notna()
        )
        g["eligible_full_numeric"] = (
            g[full_lags].notna().all(axis=1)
            & g["future_growth_change"].notna()
        )
        pieces.append(g)

    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()




## 3. Build the SIC 3674 dataset

This is the long-running cell. It downloads the SEC bulk submissions archive, filters SIC 3674 registrants, downloads Company Facts for eligible domestic issuers, and builds the model-ready dataset.

The audit CSV records why each company was included or excluded.


In [ ]:
print("SEC User-Agent:", SEC_USER_AGENT)
print("Downloading SEC submissions bulk archive...")
session = make_session()
zip_bytes = get_bytes(session, SUBMISSIONS_ZIP_URL)

print("Building full SIC 3674 registrant universe...")
universe = build_sic_universe_from_submissions(zip_bytes)
universe.to_csv(OUTPUT_DIR / "sic3674_universe.csv", index=False)

print(f"Found {len(universe):,} SIC 3674 EDGAR registrants.")
print(
    "Current ticker-bearing registrants:",
    int(universe["currently_has_ticker"].sum()),
)
print(
    universe["issuer_form_family"]
    .value_counts(dropna=False)
    .to_string()
)

company_frames = []
audit_rows = []

for i, (_, row) in enumerate(universe.iterrows(), start=1):
    cik = row["cik"]
    company = row["company"]

    # Primary numeric pipeline currently standardizes on US 10-Q/10-K.
    if row["issuer_form_family"] != "domestic_10q_10k":
        audit_rows.append(
            {
                "cik": cik,
                "company": company,
                "status": "excluded_primary",
                "reason": "non_domestic_10q_10k_form_family",
                "quarter_count": 0,
                "eligible_revenue_history_rows": 0,
                "eligible_full_numeric_rows": 0,
            }
        )
        continue

    url = COMPANYFACTS_URL.format(cik10=cik)

    try:
        cf = get_json(session, url)
        q = build_company_quarters(cf, row)
    except requests.HTTPError as exc:
        audit_rows.append(
            {
                "cik": cik,
                "company": company,
                "status": "download_error",
                "reason": f"companyfacts_http_error:{exc.response.status_code}",
                "quarter_count": 0,
                "eligible_revenue_history_rows": 0,
                "eligible_full_numeric_rows": 0,
            }
        )
        continue
    except Exception as exc:
        audit_rows.append(
            {
                "cik": cik,
                "company": company,
                "status": "processing_error",
                "reason": type(exc).__name__,
                "quarter_count": 0,
                "eligible_revenue_history_rows": 0,
                "eligible_full_numeric_rows": 0,
            }
        )
        continue

    if q.empty:
        audit_rows.append(
            {
                "cik": cik,
                "company": company,
                "status": "excluded_primary",
                "reason": "no_usable_quarterly_revenue",
                "quarter_count": 0,
                "eligible_revenue_history_rows": 0,
                "eligible_full_numeric_rows": 0,
            }
        )
        continue

    company_frames.append(q)
    print(
        f"[{i:>4}/{len(universe)}] {company[:45]:45s} "
        f"{len(q):>3} quarters"
    )

if not company_frames:
    raise RuntimeError("No company-quarter data were produced.")

quarters = pd.concat(company_frames, ignore_index=True)
quarters = (
    quarters
    .sort_values(["cik", "quarter_end"])
    .reset_index(drop=True)
)

expanded = add_lag_features(quarters)

# Update company audit using model-ready rows.
for cik, group in expanded.groupby("cik"):
    u = universe.loc[universe["cik"] == cik].iloc[0]
    audit_rows.append(
        {
            "cik": cik,
            "company": u["company"],
            "status": "included_numeric_pipeline",
            "reason": "",
            "quarter_count": len(group),
            "eligible_revenue_history_rows": int(
                group["eligible_revenue_history"].sum()
            ),
            "eligible_full_numeric_rows": int(
                group["eligible_full_numeric"].sum()
            ),
            "first_quarter": group["quarter_end"].min(),
            "last_quarter": group["quarter_end"].max(),
        }
    )

audit = pd.DataFrame(audit_rows)
audit = (
    audit.sort_values(["status", "company"])
    .drop_duplicates(["cik"], keep="last")
    .reset_index(drop=True)
)

# Model-ready table keeps all rows that at least have an eight-quarter
# revenue-growth history. Full-numeric eligibility is a separate flag.
model_rows = expanded[
    expanded["eligible_revenue_history"]
].copy()

quarters.to_csv(
    OUTPUT_DIR / "sic3674_quarterly_numeric.csv",
    index=False,
)
expanded.to_csv(
    OUTPUT_DIR / "sic3674_quarterly_with_lags.csv",
    index=False,
)
model_rows.to_csv(
    OUTPUT_DIR / "sic3674_model_rows.csv",
    index=False,
)
audit.to_csv(
    OUTPUT_DIR / "sic3674_company_audit.csv",
    index=False,
)

# Optional Parquet outputs.
try:
    quarters.to_parquet(
        OUTPUT_DIR / "sic3674_quarterly_numeric.parquet",
        index=False,
    )
    expanded.to_parquet(
        OUTPUT_DIR / "sic3674_quarterly_with_lags.parquet",
        index=False,
    )
    model_rows.to_parquet(
        OUTPUT_DIR / "sic3674_model_rows.parquet",
        index=False,
    )
    audit.to_parquet(
        OUTPUT_DIR / "sic3674_company_audit.parquet",
        index=False,
    )
except Exception as exc:
    print("Parquet export skipped:", exc)

# Optional DuckDB copy, matching the user's existing data workflow.
try:
    import duckdb

    db_path = OUTPUT_DIR / "sic3674.duckdb"
    con = duckdb.connect(str(db_path))
    con.register("universe_df", universe)
    con.register("quarters_df", quarters)
    con.register("expanded_df", expanded)
    con.register("model_rows_df", model_rows)
    con.register("audit_df", audit)

    con.execute(
        "CREATE OR REPLACE TABLE sic3674_universe AS SELECT * FROM universe_df"
    )
    con.execute(
        "CREATE OR REPLACE TABLE sic3674_quarterly_numeric AS SELECT * FROM quarters_df"
    )
    con.execute(
        "CREATE OR REPLACE TABLE sic3674_quarterly_with_lags AS SELECT * FROM expanded_df"
    )
    con.execute(
        "CREATE OR REPLACE TABLE sic3674_model_rows AS SELECT * FROM model_rows_df"
    )
    con.execute(
        "CREATE OR REPLACE TABLE sic3674_company_audit AS SELECT * FROM audit_df"
    )
    con.close()
except Exception as exc:
    print("DuckDB export skipped:", exc)

print("\n=== Expansion summary ===")
print("SIC 3674 registrants:", len(universe))
print("Companies with numeric quarters:", quarters["cik"].nunique())
print("Company-quarter observations:", len(quarters))
print(
    "Rows with 8-quarter revenue history + future target:",
    int(expanded["eligible_revenue_history"].sum()),
)
print(
    "Rows with complete 8-quarter numeric feature history + future target:",
    int(expanded["eligible_full_numeric"].sum()),
)
print("\nOutputs written to:", OUTPUT_DIR.resolve())


## 4. Inspect the resulting sample

Check these counts **before training models**. This helps catch overly aggressive eligibility rules or missing XBRL tags.


In [ ]:
from pathlib import Path
import pandas as pd

OUT = Path("/content/sic3674_output")

universe = pd.read_csv(OUT / "sic3674_universe.csv")
audit = pd.read_csv(OUT / "sic3674_company_audit.csv")
quarters = pd.read_csv(OUT / "sic3674_quarterly_numeric.csv")
model_rows = pd.read_csv(OUT / "sic3674_model_rows.csv")

print("Universe registrants:", len(universe))
print("Companies with quarterly data:", quarters["cik"].nunique())
print("Quarterly observations:", len(quarters))
print("Model-ready rows:", len(model_rows))

print("\nAudit status:")
display(audit["status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))

print("\nTop exclusion reasons:")
display(
    audit["reason"]
    .fillna("")
    .replace("", "included")
    .value_counts()
    .head(20)
    .rename_axis("reason")
    .to_frame("count")
)

print("\nModel rows preview:")
display(model_rows.head())


## 5. Optional: copy outputs to Google Drive

Use this only if you want the generated dataset to persist after the Colab runtime ends.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path("/content/sic3674_output")
destination = Path("/content/drive/MyDrive/sic3674_output")

if destination.exists():
    shutil.rmtree(destination)

shutil.copytree(source, destination)
print("Copied dataset to:", destination)


## Next research step

Do **not** immediately train the final models. First inspect `sic3674_company_audit.csv`, company counts, date ranges, missingness, and class balance.

Once those checks look reasonable, the next experiment should compare:

1. **Text-only**
2. **Fundamentals-only**
3. **Combined text + fundamentals**

using the same chronological train/validation/test periods.


## 6. Download the exact SEC filing text and extract MD&A

The numeric dataset already preserves `cik`, `source_accession`, `source_form`, and `source_filing_date`.

This stage uses those identifiers to download the **raw EDGAR submission** for that exact accession number. It then selects the main `10-Q` / `10-K` document from the filing package and extracts:

- `full_filing_text`
- `mda_text`
- downloader/extractor status fields

For 10-Q filings, MD&A is normally **Item 2**.  
For 10-K filings, MD&A is normally **Item 7**.

The extractor intentionally keeps an audit status rather than silently discarding failures. SEC filings vary across companies and years, so failed/short extractions should be inspected before modeling.


In [ ]:

import html
import re
import time
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

TEXT_OUTPUT_DIR = Path("/content/sic3674_output/filing_text")
TEXT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_REQUEST_SLEEP_SECONDS = 0.20


def make_sec_text_session() -> requests.Session:
    session = requests.Session()
    session.headers.update({
        "User-Agent": os.environ["SEC_USER_AGENT"],
        "Accept-Encoding": "gzip, deflate",
    })
    retry = Retry(
        total=5,
        backoff_factor=1.0,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET",),
    )
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session


def accession_raw_submission_url(cik, accession: str) -> str:
    cik_no_zeros = str(int(cik))
    accession = str(accession).strip()
    accession_compact = accession.replace("-", "")
    return (
        f"https://www.sec.gov/Archives/edgar/data/"
        f"{cik_no_zeros}/{accession_compact}/{accession}.txt"
    )


def split_submission_documents(raw_submission: str) -> list[dict]:
    blocks = re.findall(
        r"<DOCUMENT>(.*?)</DOCUMENT>",
        raw_submission,
        flags=re.IGNORECASE | re.DOTALL,
    )
    documents = []
    for block in blocks:
        def field(name: str) -> str:
            match = re.search(
                rf"<{name}>\s*([^\r\n<]+)",
                block,
                flags=re.IGNORECASE,
            )
            return match.group(1).strip() if match else ""

        text_match = re.search(
            r"<TEXT>(.*?)</TEXT>",
            block,
            flags=re.IGNORECASE | re.DOTALL,
        )
        documents.append({
            "type": field("TYPE").upper(),
            "sequence": field("SEQUENCE"),
            "filename": field("FILENAME"),
            "description": field("DESCRIPTION"),
            "text": text_match.group(1) if text_match else block,
        })
    return documents


def normalize_form(form: str) -> str:
    form = str(form).upper().strip()
    if form.startswith("10-Q"):
        return "10-Q"
    if form.startswith("10-K"):
        return "10-K"
    return form


def choose_primary_filing_document(documents: list[dict], expected_form: str) -> dict | None:
    target = normalize_form(expected_form)
    exact = [d for d in documents if normalize_form(d.get("type", "")) == target]
    if exact:
        return max(exact, key=lambda d: len(d.get("text", "")))

    html_candidates = [
        d for d in documents
        if str(d.get("filename", "")).lower().endswith((".htm", ".html"))
        and not str(d.get("type", "")).upper().startswith(("EX-", "GRAPHIC"))
    ]
    if html_candidates:
        return max(html_candidates, key=lambda d: len(d.get("text", "")))
    return None


def html_to_readable_text(document_text: str) -> str:
    if not document_text:
        return ""
    soup = BeautifulSoup(document_text, "lxml")
    for tag in soup(["script", "style", "noscript", "svg", "ix:header", "ix:hidden"]):
        tag.decompose()
    for tag in soup.find_all(["p", "div", "tr", "li", "h1", "h2", "h3", "h4", "br"]):
        tag.append("\n")
    text = soup.get_text("\n")
    text = html.unescape(text).replace("\xa0", " ").replace("\u200b", "")
    lines = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if line:
            lines.append(line)
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _heading_positions(text: str, pattern: str) -> list[re.Match]:
    return list(re.finditer(pattern, text, flags=re.IGNORECASE | re.MULTILINE))


def extract_mda(text: str, form: str) -> tuple[str, str]:
    form = normalize_form(form)
    if not text:
        return "", "empty_filing_text"

    if form == "10-Q":
        start_pattern = (
            r"^[ \t]*ITEM[ \t]+2[.\s:-]+"
            r"MANAGEMENT(?:'|’)?S[ \t]+DISCUSSION[ \t]+AND[ \t]+ANALYSIS\b"
        )
        end_patterns = [
            r"^[ \t]*ITEM[ \t]+3[.\s:-]+QUANTITATIVE[ \t]+AND[ \t]+QUALITATIVE[ \t]+DISCLOSURES\b",
            r"^[ \t]*ITEM[ \t]+4[.\s:-]+CONTROLS[ \t]+AND[ \t]+PROCEDURES\b",
        ]
    elif form == "10-K":
        start_pattern = (
            r"^[ \t]*ITEM[ \t]+7[.\s:-]+"
            r"MANAGEMENT(?:'|’)?S[ \t]+DISCUSSION[ \t]+AND[ \t]+ANALYSIS\b"
        )
        end_patterns = [
            r"^[ \t]*ITEM[ \t]+7A[.\s:-]+QUANTITATIVE[ \t]+AND[ \t]+QUALITATIVE[ \t]+DISCLOSURES\b",
            r"^[ \t]*ITEM[ \t]+8[.\s:-]+FINANCIAL[ \t]+STATEMENTS\b",
        ]
    else:
        return "", f"unsupported_form:{form}"

    starts = _heading_positions(text, start_pattern)
    if not starts:
        return "", "mda_start_not_found"

    all_ends = []
    for ep in end_patterns:
        all_ends.extend(_heading_positions(text, ep))
    all_ends = sorted(all_ends, key=lambda m: m.start())

    candidates = []
    for start in starts:
        following = [e for e in all_ends if e.start() > start.end()]
        if not following:
            continue
        end = following[0]
        segment = text[start.start():end.start()].strip()
        if len(segment) >= 1500:
            candidates.append(segment)

    if not candidates:
        return "", "mda_end_not_found_or_section_too_short"
    return max(candidates, key=len), "ok"


def download_and_extract_filing(session, cik, accession: str, form: str) -> dict:
    url = accession_raw_submission_url(cik, accession)
    time.sleep(TEXT_REQUEST_SLEEP_SECONDS)
    response = session.get(url, timeout=120)
    response.raise_for_status()
    docs = split_submission_documents(response.text)
    primary = choose_primary_filing_document(docs, form)
    if primary is None:
        return {
            "filing_url": url,
            "primary_filename": None,
            "full_filing_text": "",
            "mda_text": "",
            "text_status": "primary_document_not_found",
        }
    full_text = html_to_readable_text(primary["text"])
    mda_text, mda_status = extract_mda(full_text, form)
    return {
        "filing_url": url,
        "primary_filename": primary.get("filename"),
        "full_filing_text": full_text,
        "mda_text": mda_text,
        "text_status": mda_status,
    }


def add_text_to_model_rows(model_rows: pd.DataFrame, checkpoint_every: int = 100, max_rows: int | None = None) -> pd.DataFrame:
    required = {"cik", "source_accession", "source_form"}
    missing = required - set(model_rows.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    keys = (
        model_rows[["cik", "source_accession", "source_form", "source_filing_date"]]
        .dropna(subset=["cik", "source_accession", "source_form"])
        .drop_duplicates(["cik", "source_accession", "source_form"])
        .reset_index(drop=True)
    )
    if max_rows is not None:
        keys = keys.head(max_rows).copy()

    session = make_sec_text_session()
    records = []
    checkpoint_path = TEXT_OUTPUT_DIR / "filing_text_checkpoint.csv"

    for i, row in keys.iterrows():
        cik = row["cik"]
        accession = str(row["source_accession"])
        form = str(row["source_form"])
        try:
            result = download_and_extract_filing(session, cik, accession, form)
        except requests.HTTPError as exc:
            status_code = exc.response.status_code if exc.response is not None else "unknown"
            result = {
                "filing_url": accession_raw_submission_url(cik, accession),
                "primary_filename": None,
                "full_filing_text": "",
                "mda_text": "",
                "text_status": f"http_error:{status_code}",
            }
        except Exception as exc:
            result = {
                "filing_url": accession_raw_submission_url(cik, accession),
                "primary_filename": None,
                "full_filing_text": "",
                "mda_text": "",
                "text_status": f"error:{type(exc).__name__}",
            }

        records.append({
            "cik": cik,
            "source_accession": accession,
            "source_form": form,
            "source_filing_date": row.get("source_filing_date"),
            **result,
        })

        if (i + 1) % checkpoint_every == 0:
            pd.DataFrame(records).to_csv(checkpoint_path, index=False)
            print(f"Processed {i + 1:,}/{len(keys):,} unique filings")

    text_df = pd.DataFrame(records)
    text_df.to_csv(TEXT_OUTPUT_DIR / "sic3674_filing_text.csv", index=False)
    try:
        text_df.to_parquet(TEXT_OUTPUT_DIR / "sic3674_filing_text.parquet", index=False)
    except Exception as exc:
        print("Text Parquet export skipped:", exc)

    return model_rows.merge(
        text_df.drop(columns=["source_filing_date"], errors="ignore"),
        on=["cik", "source_accession", "source_form"],
        how="left",
    )


### 6a. Smoke-test the extractor first

Before downloading the whole corpus, test 20 unique filings and manually inspect the extracted MD&A sections.


In [ ]:
MODEL_PATH = Path("/content/sic3674_output/sic3674_model_rows.csv")
model_rows_for_text = pd.read_csv(MODEL_PATH)

text_smoke = add_text_to_model_rows(
    model_rows_for_text,
    max_rows=20,
    checkpoint_every=10,
)

print(text_smoke["text_status"].value_counts(dropna=False))
display(text_smoke[[
    "company", "quarter_end", "source_form", "source_accession",
    "text_status", "mda_text"
]].head(10))


### 6b. Build the full SIC 3674 text corpus

Run this after the smoke test looks good. Each unique filing is downloaded once, then joined back to all company-quarter rows.


In [ ]:
model_rows_for_text = pd.read_csv("/content/sic3674_output/sic3674_model_rows.csv")

sic3674_multimodal = add_text_to_model_rows(
    model_rows_for_text,
    max_rows=None,
    checkpoint_every=100,
)

MULTIMODAL_CSV = Path("/content/sic3674_output/sic3674_multimodal_model_rows.csv")
sic3674_multimodal.to_csv(MULTIMODAL_CSV, index=False)

try:
    sic3674_multimodal.to_parquet(
        "/content/sic3674_output/sic3674_multimodal_model_rows.parquet",
        index=False,
    )
except Exception as exc:
    print("Multimodal Parquet export skipped:", exc)

print("Saved:", MULTIMODAL_CSV)
display(sic3674_multimodal["text_status"].value_counts(dropna=False).rename_axis("status").to_frame("count"))
ok = sic3674_multimodal["text_status"].eq("ok")
print(f"Rows with extracted MD&A: {ok.sum():,} / {len(sic3674_multimodal):,}")


## 7. MD&A quality checks

Review extraction success by form/year, MD&A length, and random examples before using the text in TF-IDF or FinBERT.


In [ ]:
qc = sic3674_multimodal.copy()
qc["mda_chars"] = qc["mda_text"].fillna("").str.len()
qc["mda_words"] = qc["mda_text"].fillna("").str.split().str.len()
qc["filing_year"] = pd.to_datetime(qc["source_filing_date"], errors="coerce").dt.year

print("Status by form:")
display(pd.crosstab(qc["source_form"], qc["text_status"], margins=True))

print("MD&A length for successful extractions:")
display(qc.loc[qc["text_status"] == "ok", ["mda_chars", "mda_words"]].describe())

print("Success rate by filing year:")
year_qc = (
    qc.assign(extraction_ok=qc["text_status"].eq("ok"))
      .groupby("filing_year", dropna=False)["extraction_ok"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "success_rate"})
)
display(year_qc.tail(20))


## 8. Outputs for the three central experiments

After this stage each eligible observation contains both modalities:

1. **Text-only:** `mda_text` → TF-IDF or FinBERT → classifier
2. **Fundamentals-only:** lagged numeric features → classifier
3. **Combined:** text representation + numeric fundamentals → classifier

Keep the chronological train/validation/test periods identical across all three.
